In [1]:
import pandas as pd

In [6]:
df_a100 = pd.read_csv("../results/benchmark_bsrmv_A100.csv")
df_h100 = pd.read_csv("../results/benchmark_bsrmv_H100.csv")
df_nnz = pd.read_csv("../results/bsrmv_nnz.csv")

In [3]:
df_a100["gpu"] = "A100"
df_h100["gpu"] = "H100"
df = pd.concat([df_a100, df_h100], ignore_index=True)

In [4]:
df["dtype"] = df["dtype"].str.replace("torch.float", "float")
df["elements"] = df.apply(
    lambda row: (
        2 * 4 ** row["mesh m"] if row["dimension"] == "2D" else 6 * 8 ** row["mesh m"]
    ),
    axis=1,
)
df.loc[
    (df["algorithm"] == "bsr_block_by_block") & (df["DOFs per element"] > 5),
    ["time [ms]"],
] = None

In [13]:
df = df.merge(right=df_nnz, on=["dimension", "mesh m", "degree", "DOFs"])

In [14]:
df

,dimension,mesh m,dtype,DOFs,DOFs per element,degree,algorithm,time [ms],error norm,gpu,elements,nnz
0,2D,11,float32,25165824,3,1,bsr_cusparse,4.694569,0.000000e+00,A100,8388608,301916160
1,2D,11,float32,25165824,3,1,bsr_row_per_thread_irrespective,1.029519,3.896300e-03,A100,8388608,301916160
2,2D,11,float32,25165824,3,1,bsr_column_by_column,6.539622,3.896300e-03,A100,8388608,301916160
3,2D,11,float32,25165824,3,1,bsr_block_by_block,6.527478,3.896300e-03,A100,8388608,301916160
4,2D,11,float32,25165824,3,1,bsr_row_per_thread,8.272394,3.896300e-03,A100,8388608,301916160
...,...,...,...,...,...,...,...,...,...,...,...,...
303,3D,5,float64,3932160,20,3,bsr_column_by_column,2.074609,8.459305e-14,H100,196608,388300800
304,3D,5,float64,3932160,20,3,bsr_block_by_block,NaN,8.459305e-14,H100,196608,388300800
305,3D,5,float64,3932160,20,3,bsr_row_per_thread,1.424208,8.459305e-14,H100,196608,388300800
306,3D,5,float64,3932160,20,3,bsr_row_per_thread_sync,1.402998,8.459305e-14,H100,196608,388300800


In [17]:
def tput(row):
    dofs = row["DOFs"]
    dofs_per_el = row["DOFs per element"]
    nnz = row["nnz"]
    if row["dtype"] == "float32":
        bytes_per_value = 4
    elif row["dtype"] == "float64":
        bytes_per_value = 8
    else:
        raise ValueError(f"Unknown dtype: {row['dtype']}")
    bytes_per_index = 4

    time_s = row["time [ms]"] * 1e-3
    if time_s == 0 or pd.isna(time_s):
        return None

    if "csr" in row["algorithm"]:
        matrix_size = (
            nnz * bytes_per_value + nnz * bytes_per_index + (dofs + 1) * bytes_per_index
        )
    else:
        matrix_size = (
            nnz * bytes_per_value
            + (nnz // dofs_per_el // dofs_per_el) * bytes_per_index
            + (dofs // dofs_per_el + 1) * bytes_per_index
        )
    vector_size = dofs * bytes_per_value
    total_bytes = matrix_size + 2 * vector_size

    return total_bytes / time_s / 1e9  # GB/s

In [18]:
df["tput [GB/s]"] = df.apply(tput, axis=1)
df

,dimension,mesh m,dtype,DOFs,DOFs per element,degree,algorithm,time [ms],error norm,gpu,elements,nnz,tput [GB/s]
0,2D,11,float32,25165824,3,1,bsr_cusparse,4.694569,0.000000e+00,A100,8388608,301916160,335.862695
1,2D,11,float32,25165824,3,1,bsr_row_per_thread_irrespective,1.029519,3.896300e-03,A100,8388608,301916160,1531.521123
2,2D,11,float32,25165824,3,1,bsr_column_by_column,6.539622,3.896300e-03,A100,8388608,301916160,241.104238
3,2D,11,float32,25165824,3,1,bsr_block_by_block,6.527478,3.896300e-03,A100,8388608,301916160,241.552805
4,2D,11,float32,25165824,3,1,bsr_row_per_thread,8.272394,3.896300e-03,A100,8388608,301916160,190.601481
...,...,...,...,...,...,...,...,...,...,...,...,...,...
303,3D,5,float64,3932160,20,3,bsr_column_by_column,2.074609,8.459305e-14,H100,196608,388300800,1529.922055
304,3D,5,float64,3932160,20,3,bsr_block_by_block,NaN,8.459305e-14,H100,196608,388300800,NaN
305,3D,5,float64,3932160,20,3,bsr_row_per_thread,1.424208,8.459305e-14,H100,196608,388300800,2228.600928
306,3D,5,float64,3932160,20,3,bsr_row_per_thread_sync,1.402998,8.459305e-14,H100,196608,388300800,2262.290900


In [26]:
df[df["gpu"] == "A100"].pivot_table(
    values="tput [GB/s]",
    index=["dimension", "degree"],
    columns="algorithm",
    aggfunc="max",
)

algorithm         bsr_block_by_block  bsr_column_by_column  bsr_cusparse  \
dimension degree                                                           
2D        1               456.480013            457.729910    471.709357   
          2                      NaN           1000.730926    844.408902   
          3                      NaN           1169.226104    627.565038   
          4                      NaN           1230.403508   1205.156766   
          5                      NaN            877.198850    683.357241   
3D        1               834.976906            891.801943   1307.794200   
          2                      NaN           1221.224925    657.392268   
          3                      NaN            888.387031    625.584042   

algorithm         bsr_row_per_thread  bsr_row_per_thread_irrespective  \
dimension degree                                                        
2D        1               282.205474                      1545.870738   
          2               669.231847                      1555.669085   
          3              1142.592172                      1579.470377   
          4              1319.804829                      1467.203469   
          5              1442.476391                      1475.007561   
3D        1               500.696003                      1560.833536   
          2              1143.892252                      1565.946844   
          3              1407.182405                      1553.851392   

algorithm         bsr_row_per_thread_sync  csr_cusparse  
dimension degree                                         
2D        1                    221.145026   1448.781672  
          2                    654.500358   1479.792817  
          3                   1128.446394   1536.973761  
          4                   1486.022257   1553.008551  
          5                   1524.342520   1616.540117  
3D        1                    344.351156   1477.536672  
          2                   1143.970151   1581.734312  
          3                   1576.770709   1601.431092